# 1 Импорт библиотек и функций

## 1.1 Библиотеки

In [3]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from datetime import datetime
from aiomoex import get_board_candles
import asyncio
import aiohttp
from statsmodels.graphics.tsaplots import plot_acf
import ta
from sklearn.ensemble import RandomForestRegressor
from typing import List, Dict
import os

## 1.2 Функции

In [5]:
def add_targets(stock_data, horizon_days=7):
    """
    Добавляет целевые переменные для прогнозирования
    """
    
    for ticker, df in stock_data.items():
        if df.empty:
            continue
            
        df_sorted = df.sort_values('begin')
        
        # Создаем копию данных
        df_merged = df_sorted.copy()
        
        # Добавляем будущую дату (ровно +horizon_days дней)
        df_merged['future_date'] = df_merged['begin'] + pd.Timedelta(days=horizon_days)
        
        # Создаем словарь с ценами по датам
        price_dict = df_sorted.set_index('begin')['close'].to_dict()
        
        # Добавляем будущую цену по точному соответствию дат
        df_merged['future_price'] = df_merged['future_date'].map(price_dict)
        
        # Вычисляем целевые переменные
        df_merged['target_price_change'] = (
            (df_merged['future_price'] / df_merged['close'] - 1) * 100
        )
        
        df_merged['target_class'] = df_merged['target_price_change'].apply(
            lambda x: 'L' if x < -1 else ('H' if x > 1 else 'N')
        )
        
        # Удаляем временные колонки
        df_merged = df_merged.drop(['future_date', 'future_price'], axis=1)
        
        stock_data[ticker] = df_merged
    
    return stock_data
    
def clean_generated_features(stock_data):
    """Удаляет строки с пропусками после генерации признаков"""
    cleaned_data = {}
    
    for ticker, df in stock_data.items():
        df_clean = df.dropna().copy()
        cleaned_data[ticker] = df_clean
    
    return cleaned_data

def save_features_to_csv(stock_data: Dict[str, pd.DataFrame], save_dir: str = './features_data'):
    """
    Сохраняет данные со сгенерированными признаками в CSV файлы
    """
    
    # Создаем директорию если не существует
    os.makedirs(save_dir, exist_ok=True)
    
    # Сохраняем каждый датафрейм в отдельный файл
    for ticker, df in stock_data.items():
        if not df.empty:
            # Создаем копию датафрейма
            df_to_save = df.copy()
            
            # Убираем указанные признаки
            columns_to_drop = ['open', 'high', 'low', 'value', 'volume', 'ticker']
            
            # Удаляем только те столбцы, которые существуют в датафрейме
            existing_columns_to_drop = [col for col in columns_to_drop if col in df_to_save.columns]
            df_to_save = df_to_save.drop(columns=existing_columns_to_drop)
            
            filename = f"stocks_features_{ticker}.csv"
            filepath = os.path.join(save_dir, filename)
            
            df_to_save.to_csv(filepath, index=False, encoding='utf-8')
            print(f"✅ {ticker}: обработан и сохранен ({len(df)} строк), признаков: {len(df_to_save.columns)}")
        else:
            print(f"❌ {ticker}: не удалось обработать")

In [6]:
def process_and_save_features(data_path: str, tickers: List[str], horizon_days: int = 7, save_dir: str = './features_data', end_date: str = None):
    """
    Основная функция для обработки данных и генерации признаков
    
    Args:
        data_path: Путь к папке с данными
        tickers: Список тикеров для обработки
        horizon_days: Горизонт прогнозирования в днях
        save_dir: Директория для сохранения результатов
        end_date: Верхняя граница даты в формате 'YYYY-MM-DD' (например, '2025-10-01'). 
                 Если None - фильтрация не применяется
    """
    
    stock_data = {}
    
    # Загружаем данные из CSV файлов
    for ticker in tickers:
        filename = f"stocks_{ticker}.csv"
        filepath = os.path.join(data_path, filename)
        
        try:
            df = pd.read_csv(filepath)
            df['begin'] = pd.to_datetime(df['begin'])
            stock_data[ticker] = df
        except FileNotFoundError:
            print(f"⚠️ Файл не найден: {filepath}")
            stock_data[ticker] = pd.DataFrame()
        except Exception as e:
            print(f"❌ Ошибка загрузки {ticker}: {e}")
            stock_data[ticker] = pd.DataFrame()
    
    # Генерируем признаки
    stock_data = generate_features_optimized(stock_data)
    
    # Добавляем целевые переменные
    stock_data = add_targets(stock_data, horizon_days=horizon_days)
    
    # Очищаем данные от пропусков
    stock_data = clean_generated_features(stock_data)
    
    # Применяем фильтрацию по дате если указана end_date
    if end_date:
        end_date_dt = pd.to_datetime(end_date)
        filtered_stock_data = {}
        for ticker, df in stock_data.items():
            if not df.empty:
                filtered_df = df[df['begin'] < end_date_dt].copy()
                filtered_stock_data[ticker] = filtered_df
                print(f"📅 {ticker}: отфильтровано {len(df) - len(filtered_df)} строк, осталось {len(filtered_df)}")
            else:
                filtered_stock_data[ticker] = df
        stock_data = filtered_stock_data
    
    # Сохраняем результаты
    save_features_to_csv(stock_data, save_dir=save_dir)
    
    # Сводная статистика
    processed_tickers = [ticker for ticker, df in stock_data.items() if not df.empty]
    total_rows = sum(len(df) for df in stock_data.values() if not df.empty)
    
    print(f"\n📊 Обработка завершена! Обработано {len(processed_tickers)} акций, всего строк: {total_rows}")
    
    if end_date:
        print(f"📅 Данные отфильтрованы по дате: строго до {end_date}")
    
    return stock_data

In [10]:
VALID_INTERVALS = {
    1: 1,    # 1 минута
    10: 10,  # 10 минут
    60: 60,  # 1 час
    120: 120, # 2 часа
    240: 240, # 4 часа
    24: 24,   # 1 день
    7: 7,     # 1 неделя
    31: 31    # 1 месяц
}

In [12]:
async def get_moex_data(tickers: list[str], start_date: str, end_date: str, interval: int = 24) -> dict[str, pd.DataFrame]:
    if interval not in VALID_INTERVALS:
        raise ValueError(f"Неверный интервал. Допустимые значения: {list(VALID_INTERVALS.keys())}")
    
    async with aiohttp.ClientSession() as session:
        coros = [fetch_ticker_data(session, ticker, interval, start_date, end_date) for ticker in tickers]
        stock_data = await asyncio.gather(*coros)
    
    stock_data_dict = {k: v for d in stock_data for k, v in d.items()}
    return stock_data_dict

In [14]:
async def fetch_ticker_data(session: aiohttp.ClientSession, ticker: str, interval: int, start_date: str, end_date: str) -> dict[str, pd.DataFrame]:
    try:
        # Для интервалов 120 и 240 парсим с интервалом 60 минут
        actual_interval = interval
        if interval in [120, 240]:
            actual_interval = 60
            
        res = await get_board_candles(session, ticker, actual_interval, start_date, end_date)
        if res:
            df = pd.DataFrame(res)
            
            if 'end' in df.columns:
                df = df.drop('end', axis=1)
            
            df['begin'] = pd.to_datetime(df['begin'])
            
            # Агрегация для интервалов 120 и 240 минут
            if interval == 120:
                # Агрегируем в 2-часовые интервалы, исключая 9:00
                df = df[df['begin'].dt.time != pd.Timestamp('09:00:00').time()]
                df = _aggregate_to_2h(df)
                
            elif interval == 240:
                # Агрегируем в 4-часовые интервалы, исключая 9:00
                df = df[df['begin'].dt.time != pd.Timestamp('09:00:00').time()]
                df = _aggregate_to_4h(df)
            
            if interval in [24, 7, 31]:
                df['begin'] = df['begin'].dt.normalize()

            if 'begin' in df.columns:
                cols = ['begin'] + [col for col in df.columns if col != 'begin']
                df = df[cols]
            
            df['ticker'] = ticker
            return {ticker: df}
        else:
            return {ticker: pd.DataFrame()}
    except Exception as e:
        print(f'Ошибка парсинга. Не удалось получить данные для {ticker}, {e}')
        return {ticker: pd.DataFrame()}

def _aggregate_to_2h(df):
    """Агрегирует часовые данные в 2-часовые интервалы только для основной сессии"""
    df = df.sort_values('begin')
    
    # Ограничиваем основной торговой сессией (10:00-18:45)
    df = df[(df['begin'].dt.time >= pd.Timestamp('10:00:00').time()) & 
            (df['begin'].dt.time <= pd.Timestamp('18:45:00').time())]
    
    df['time_group'] = df['begin'].dt.floor('2h')  
    
    odd_hour_mask = df['time_group'].dt.hour % 2 == 1
    df.loc[odd_hour_mask, 'time_group'] = df.loc[odd_hour_mask, 'time_group'] - pd.Timedelta(hours=1)
    
    # Фильтруем только нужные интервалы (10:00, 12:00, 14:00, 16:00, 18:00)
    df = df[df['time_group'].dt.hour.isin([10, 12, 14, 16, 18])]
    
    aggregated = df.groupby('time_group').agg({
        'open': 'first',
        'close': 'last',
        'high': 'max',
        'low': 'min',
        'volume': 'sum',
        'value': 'sum'
    }).reset_index()
    
    aggregated = aggregated.rename(columns={'time_group': 'begin'})
    return aggregated

def _aggregate_to_4h(df):
    """Агрегирует часовые данные в 4-часовые интервалы только для основной сессии"""
    df = df.sort_values('begin')
    
    # Ограничиваем основной торговой сессией (10:00-18:45)
    df = df[(df['begin'].dt.time >= pd.Timestamp('10:00:00').time()) & 
            (df['begin'].dt.time <= pd.Timestamp('18:45:00').time())]
    
    df['time_group'] = df['begin'].dt.floor('4h')  
    
    # Корректируем группы чтобы они начинались с 10:00, 14:00, 18:00
    hour_mod = df['time_group'].dt.hour % 4
    df['time_group'] = df['time_group'] - pd.to_timedelta(hour_mod, unit='h')
    
    # Фильтруем только нужные интервалы (10:00, 14:00, 18:00)
    df = df[df['time_group'].dt.hour.isin([10, 14, 18])]
    
    aggregated = df.groupby('time_group').agg({
        'open': 'first',
        'close': 'last',
        'high': 'max',
        'low': 'min',
        'volume': 'sum',
        'value': 'sum'
    }).reset_index()
    
    aggregated = aggregated.rename(columns={'time_group': 'begin'})
    return aggregated

In [16]:
async def save_moex_data_to_csv(
    tickers: List[str],
    start_date: str = '2022-05-01',
    end_date: str = '2025-10-01', 
    interval: int = 24,
    save_dir: str = './data'
) -> None:
    """
    Сохраняет данные MOEX в CSV файлы
    
    Args:
        tickers: Список тикеров
        start_date: Начальная дата в формате 'YYYY-MM-DD'
        end_date: Конечная дата в формате 'YYYY-MM-DD'
        interval: Интервал данных (1, 10, 60, 24, 7, 31)
        save_dir: Директория для сохранения файлов
    """
    
    stock_data = await get_moex_data(
        tickers=tickers,
        start_date=start_date,
        end_date=end_date,
        interval=interval
    )
    
    os.makedirs(save_dir, exist_ok=True)
    
    for ticker, df in stock_data.items():
        if not df.empty:
            filename = f"stocks_{ticker}.csv"
            filepath = os.path.join(save_dir, filename)
            
            df.to_csv(filepath, index=False, encoding='utf-8')
            print(f"Данные для {ticker} сохранены в {filepath}")
        else:
            print(f"Нет данных для тикера {ticker}")

# 2 Подготовка данных

In [25]:
stock_data = process_and_save_features(
    data_path='../../data/stock_prices',  
    tickers=tickers,          
    horizon_days=7,           
    save_dir='../../data/stock_features_data',
    end_date="2025-10-01"
)

✅ SBER: 83 признаков (34 базовых + 49 производных)
✅ TCSG: 83 признаков (34 базовых + 49 производных)
✅ GAZP: 83 признаков (34 базовых + 49 производных)
✅ LKOH: 83 признаков (34 базовых + 49 производных)
✅ ROSN: 83 признаков (34 базовых + 49 производных)
📅 SBER: отфильтровано 40 строк, осталось 4178
📅 TCSG: отфильтровано 0 строк, осталось 2853
📅 GAZP: отфильтровано 40 строк, осталось 4178
📅 LKOH: отфильтровано 40 строк, осталось 4178
📅 ROSN: отфильтровано 40 строк, осталось 4178
✅ SBER: обработан и сохранен (4178 строк), признаков: 87
✅ TCSG: обработан и сохранен (2853 строк), признаков: 87
✅ GAZP: обработан и сохранен (4178 строк), признаков: 87
✅ LKOH: обработан и сохранен (4178 строк), признаков: 87
✅ ROSN: обработан и сохранен (4178 строк), признаков: 87

📊 Обработка завершена! Обработано 5 акций, всего строк: 19565
📅 Данные отфильтрованы по дате: строго до 2025-10-01


In [23]:
tickers = [
    'SBER', 'TCSG', 'GAZP', 'LKOH', 'ROSN'
]

### 2.2.1 Считвание и проверка

In [19]:
data_features_sber = pd.read_csv("../../data/stock_features_data/stocks_features_SBER.csv")
data_features_sber.head(10)

,begin,close,MA_20,MA_50,MA_90,MA_200,EMA_20,EMA_50,EMA_90,EMA_200,...,DISTANCE_EMA_20,DISTANCE_EMA_50,DISTANCE_EMA_90,DISTANCE_EMA_200,DISTANCE_WMA_20,DISTANCE_WMA_50,DISTANCE_WMA_90,DISTANCE_WMA_200,target_price_change,target_class
0,2022-07-04 10:00:00,129.63,132.6825,133.4456,127.631444,124.63330,131.580771,131.515338,129.399081,126.786750,...,-0.014826,-0.014335,0.001785,0.022425,-0.006353,-0.030611,-0.011263,0.025350,1.357710,H
1,2022-07-04 12:00:00,128.30,132.1140,133.5142,127.749222,124.64805,131.268317,131.389207,129.374651,126.804110,...,-0.022613,-0.023512,-0.008307,0.011797,-0.013391,-0.039106,-0.021517,0.014538,0.202650,N
2,2022-07-04 14:00:00,128.48,131.5480,133.5738,127.861222,124.66815,131.002763,131.275087,129.354770,126.823306,...,-0.019257,-0.021292,-0.006763,0.013063,-0.009370,-0.036334,-0.020265,0.015655,0.046700,N
3,2022-07-04 16:00:00,131.29,131.1130,133.6196,127.999444,124.71110,131.030119,131.275672,129.397764,126.874394,...,0.001983,0.000109,0.014623,0.034803,0.012488,-0.014595,0.000588,0.037328,-2.726788,L
4,2022-07-04 18:00:00,131.43,130.6850,133.6224,128.145111,124.75225,131.068203,131.281725,129.442903,126.926421,...,0.002760,0.001129,0.015351,0.035482,0.013331,-0.012908,0.001080,0.037886,-3.522788,L
5,2022-07-05 10:00:00,133.14,130.4080,133.7202,128.276333,124.80090,131.265517,131.354618,129.525002,126.997278,...,0.014280,0.013592,0.027910,0.048369,0.024668,0.000077,0.013258,0.050697,-4.266186,L
6,2022-07-05 12:00:00,133.90,130.2120,133.8166,128.423556,124.84785,131.516420,131.454462,129.622131,127.075880,...,0.018124,0.018604,0.033003,0.053701,0.027887,0.005732,0.018084,0.055940,-5.436893,L
7,2022-07-05 14:00:00,134.80,130.1600,133.9186,128.584222,124.90075,131.829142,131.585692,129.737060,127.163710,...,0.022536,0.024427,0.039025,0.060051,0.031336,0.012199,0.023836,0.062208,-5.956973,L
8,2022-07-05 16:00:00,134.20,130.1200,134.0092,128.742222,124.95030,132.054938,131.688238,129.836099,127.243604,...,0.016244,0.019074,0.033611,0.054670,0.023732,0.007610,0.018324,0.056710,-3.658718,L
9,2022-07-05 18:00:00,133.50,130.1550,134.0972,128.893333,124.99380,132.192563,131.759303,129.917389,127.314544,...,0.009890,0.013211,0.027576,0.048584,0.015897,0.002505,0.012209,0.050494,-3.370787,L


In [130]:
data_features_sber.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3665 entries, 0 to 3664
Data columns (total 24 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   begin                3665 non-null   object 
 1   close                3665 non-null   float64
 2   MA_90                3665 non-null   float64
 3   RSI_14               3665 non-null   float64
 4   RSI_90               3665 non-null   float64
 5   MOM_10               3665 non-null   float64
 6   ATR_14               3665 non-null   float64
 7   VOLATILITY_20        3665 non-null   float64
 8   VOLATILITY_50        3665 non-null   float64
 9   VOLUME_RATIO_20      3665 non-null   float64
 10  MACD_SIGNAL          3665 non-null   float64
 11  MACD_HISTOGRAM       3665 non-null   float64
 12  MA_20                3665 non-null   float64
 13  MA_50                3665 non-null   float64
 14  RSI_7                3665 non-null   float64
 15  MOM_5                3665 non-null   f

# Лучшние варианты

## Для газпрома

In [ ]:
def generate_features_optimized(stock_data: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """
    Генерирует финальный набор признаков с разностями между разными типами скользящих средних
    """
    
    processed_data = {}
    
    for ticker, df in stock_data.items():
        if df.empty:
            processed_data[ticker] = df
            continue
            
        df_processed = df.copy().sort_values('begin')
        
        # ===== 1. СКОЛЬЗЯЩИЕ СРЕДНИЕ РАЗНЫХ ТИПОВ =====
        # Простые скользящие средние
        df_processed['MA_20'] = df_processed['close'].rolling(window=20).mean()
        df_processed['MA_50'] = df_processed['close'].rolling(window=50).mean()
        df_processed['MA_90'] = df_processed['close'].rolling(window=90).mean()
        
        # Экспоненциальные скользящие средние
        df_processed['EMA_20'] = df_processed['close'].ewm(span=20).mean()
        df_processed['EMA_50'] = df_processed['close'].ewm(span=50).mean()
        df_processed['EMA_90'] = df_processed['close'].ewm(span=90).mean()
        
        # Взвешенные скользящие средние
        df_processed['WMA_20'] = df_processed['close'].rolling(window=20).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_50'] = df_processed['close'].rolling(window=50).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_90'] = df_processed['close'].rolling(window=90).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        
        # ===== 2. RSI (3 окна) =====
        df_processed['RSI_7'] = ta.momentum.RSIIndicator(df_processed['close'], window=7).rsi()
        df_processed['RSI_14'] = ta.momentum.RSIIndicator(df_processed['close'], window=14).rsi()
        df_processed['RSI_90'] = ta.momentum.RSIIndicator(df_processed['close'], window=90).rsi()
        
        # ===== 3. МОМЕНТУМ (3 окна) =====
        df_processed['MOM_10'] = ta.momentum.ROCIndicator(df_processed['close'], window=5).roc()
        df_processed['MOM_20'] = ta.momentum.ROCIndicator(df_processed['close'], window=10).roc()
        df_processed['MOM_50'] = ta.momentum.ROCIndicator(df_processed['close'], window=20).roc()
        
        # ===== 4. ВОЛАТИЛЬНОСТЬ (3 окна) =====
        returns = df_processed['close'].pct_change()
        df_processed['VOLATILITY_10'] = returns.rolling(window=10).std() * np.sqrt(252)
        df_processed['VOLATILITY_20'] = returns.rolling(window=20).std() * np.sqrt(252)
        df_processed['VOLATILITY_50'] = returns.rolling(window=50).std() * np.sqrt(252)
        
        # ===== 5. ATR (2 окна) =====
        df_processed['ATR_7'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=7
        ).average_true_range()
        df_processed['ATR_14'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=14
        ).average_true_range()
        
        # ===== 6. MACD (полный комплекс) =====
        macd = ta.trend.MACD(df_processed['close'])
        df_processed['MACD'] = macd.macd()
        df_processed['MACD_SIGNAL'] = macd.macd_signal()
        df_processed['MACD_HISTOGRAM'] = macd.macd_diff()
        
        # ===== 7. ОБЪЕМ (2 признака) =====
        df_processed['VOLUME_RATIO_20'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=20).mean()
        )
        df_processed['VOLUME_RATIO_50'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=50).mean()
        )
        
        # ===== 9. ПРОЦЕНТНЫЕ ИЗМЕНЕНИЯ =====
        df_processed['RETURN_1D'] = df_processed['close'].pct_change(5)    # Дневное изменение
        df_processed['RETURN_1W'] = df_processed['close'].pct_change(20)    # Недельное изменение (5 торговых дней)
        
        # ===== 10. РАЗНОСТИ МЕЖДУ РАЗНЫМИ ТИПАМИ СКОЛЬЗЯЩИХ СРЕДНИХ =====
        
        # Разности между MA и EMA (разная чувствительность)
        df_processed['MA_EMA_DIFF_20'] = df_processed['MA_20'] - df_processed['EMA_20']
        df_processed['MA_EMA_DIFF_50'] = df_processed['MA_50'] - df_processed['EMA_50']
        df_processed['MA_EMA_DIFF_90'] = df_processed['MA_90'] - df_processed['EMA_90']
        
        # Разности между MA и WMA (разные веса)
        df_processed['MA_WMA_DIFF_20'] = df_processed['MA_20'] - df_processed['WMA_20']
        df_processed['MA_WMA_DIFF_50'] = df_processed['MA_50'] - df_processed['WMA_50']
        df_processed['MA_WMA_DIFF_90'] = df_processed['MA_90'] - df_processed['WMA_90']
        
        # Разности между EMA и WMA
        df_processed['EMA_WMA_DIFF_20'] = df_processed['EMA_20'] - df_processed['WMA_20']
        df_processed['EMA_WMA_DIFF_50'] = df_processed['EMA_50'] - df_processed['WMA_50']
        df_processed['EMA_WMA_DIFF_90'] = df_processed['EMA_90'] - df_processed['WMA_90']
        
        # Отношения между разными типами скользящих средних
        df_processed['MA_EMA_RATIO_20'] = df_processed['MA_20'] / df_processed['EMA_20']
        df_processed['MA_EMA_RATIO_50'] = df_processed['MA_50'] / df_processed['EMA_50']
        df_processed['MA_EMA_RATIO_90'] = df_processed['MA_90'] / df_processed['EMA_90']
        
        # Разности между MA разных периодов
        df_processed['MA_DIFF_20_50'] = df_processed['MA_20'] - df_processed['MA_50']
        df_processed['MA_DIFF_20_90'] = df_processed['MA_20'] - df_processed['MA_90']
        df_processed['MA_DIFF_50_90'] = df_processed['MA_50'] - df_processed['MA_90']
        
        # Разности между EMA разных периодов
        df_processed['EMA_DIFF_20_50'] = df_processed['EMA_20'] - df_processed['EMA_50']
        df_processed['EMA_DIFF_20_90'] = df_processed['EMA_20'] - df_processed['EMA_90']
        df_processed['EMA_DIFF_50_90'] = df_processed['EMA_50'] - df_processed['EMA_90']
        
        # Разности между WMA разных периодов
        df_processed['WMA_DIFF_20_50'] = df_processed['WMA_20'] - df_processed['WMA_50']
        df_processed['WMA_DIFF_20_90'] = df_processed['WMA_20'] - df_processed['WMA_90']
        df_processed['WMA_DIFF_50_90'] = df_processed['WMA_50'] - df_processed['WMA_90']
        
        # Отношения между MA разных периодов
        df_processed['MA_RATIO_20_50'] = df_processed['MA_20'] / df_processed['MA_50']
        df_processed['MA_RATIO_20_90'] = df_processed['MA_20'] / df_processed['MA_90']
        df_processed['MA_RATIO_50_90'] = df_processed['MA_50'] / df_processed['MA_90']
        
        # Расстояния до разных типов скользящих средних
        df_processed['DISTANCE_MA_20'] = (df_processed['close'] - df_processed['MA_20']) / df_processed['MA_20']
        df_processed['DISTANCE_MA_50'] = (df_processed['close'] - df_processed['MA_50']) / df_processed['MA_50']
        df_processed['DISTANCE_MA_90'] = (df_processed['close'] - df_processed['MA_90']) / df_processed['MA_90']
        
        df_processed['DISTANCE_EMA_20'] = (df_processed['close'] - df_processed['EMA_20']) / df_processed['EMA_20']
        df_processed['DISTANCE_EMA_50'] = (df_processed['close'] - df_processed['EMA_50']) / df_processed['EMA_50']
        df_processed['DISTANCE_EMA_90'] = (df_processed['close'] - df_processed['EMA_90']) / df_processed['EMA_90']
        
        df_processed['DISTANCE_WMA_20'] = (df_processed['close'] - df_processed['WMA_20']) / df_processed['WMA_20']
        df_processed['DISTANCE_WMA_50'] = (df_processed['close'] - df_processed['WMA_50']) / df_processed['WMA_50']
        df_processed['DISTANCE_WMA_90'] = (df_processed['close'] - df_processed['WMA_90']) / df_processed['WMA_90']
        
        processed_data[ticker] = df_processed
        
    return processed_data

Чем больше признаков тем лучше на газпроме и хуже на сбере 

Положтельный r2 на бустинге

In [ ]:
def generate_features_optimized(stock_data: Dict[str, pd.DataFrame]) -> Dict[str, pd.DataFrame]:
    """
    Генерирует финальный набор признаков с разностями между разными типами скользящих средних
    """
    
    processed_data = {}
    
    for ticker, df in stock_data.items():
        if df.empty:
            processed_data[ticker] = df
            continue
            
        df_processed = df.copy().sort_values('begin')
        
        # ===== 1. СКОЛЬЗЯЩИЕ СРЕДНИЕ РАЗНЫХ ТИПОВ =====
        # Простые скользящие средние
        df_processed['MA_20'] = df_processed['close'].rolling(window=20).mean()
        df_processed['MA_50'] = df_processed['close'].rolling(window=50).mean()
        df_processed['MA_90'] = df_processed['close'].rolling(window=90).mean()
        df_processed['MA_200'] = df_processed['close'].rolling(window=200).mean()  # Долгосрочный тренд
        
        # Экспоненциальные скользящие средние
        df_processed['EMA_20'] = df_processed['close'].ewm(span=20).mean()
        df_processed['EMA_50'] = df_processed['close'].ewm(span=50).mean()
        df_processed['EMA_90'] = df_processed['close'].ewm(span=90).mean()
        df_processed['EMA_200'] = df_processed['close'].ewm(span=200).mean()  # Долгосрочный тренд
        
        # Взвешенные скользящие средние
        df_processed['WMA_20'] = df_processed['close'].rolling(window=20).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_50'] = df_processed['close'].rolling(window=50).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_90'] = df_processed['close'].rolling(window=90).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_200'] = df_processed['close'].rolling(window=200).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )  # Долгосрочный тренд
        
        # ===== 2. RSI (3 окна) =====
        df_processed['RSI_7'] = ta.momentum.RSIIndicator(df_processed['close'], window=7).rsi()
        df_processed['RSI_14'] = ta.momentum.RSIIndicator(df_processed['close'], window=14).rsi()
        df_processed['RSI_90'] = ta.momentum.RSIIndicator(df_processed['close'], window=90).rsi()
        
        # ===== 3. МОМЕНТУМ (3 окна) =====
        df_processed['MOM_10'] = ta.momentum.ROCIndicator(df_processed['close'], window=5).roc()
        df_processed['MOM_20'] = ta.momentum.ROCIndicator(df_processed['close'], window=10).roc()
        df_processed['MOM_50'] = ta.momentum.ROCIndicator(df_processed['close'], window=20).roc()
        df_processed['MOM_200'] = ta.momentum.ROCIndicator(df_processed['close'], window=200).roc()  # Долгосрочный момент
        
        # ===== 4. ВОЛАТИЛЬНОСТЬ (3 окна) =====
        returns = df_processed['close'].pct_change()
        df_processed['VOLATILITY_10'] = returns.rolling(window=10).std() * np.sqrt(252)
        df_processed['VOLATILITY_20'] = returns.rolling(window=20).std() * np.sqrt(252)
        df_processed['VOLATILITY_50'] = returns.rolling(window=50).std() * np.sqrt(252)
        df_processed['VOLATILITY_200'] = returns.rolling(window=200).std() * np.sqrt(252)  # Долгосрочная волатильность
        
        # ===== 5. ATR (2 окна) =====
        df_processed['ATR_7'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=7
        ).average_true_range()
        df_processed['ATR_14'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=14
        ).average_true_range()
        df_processed['ATR_200'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=200
        ).average_true_range()  # Долгосрочный ATR
        
        # ===== 6. MACD (полный комплекс) =====
        macd = ta.trend.MACD(df_processed['close'])
        df_processed['MACD'] = macd.macd()
        df_processed['MACD_SIGNAL'] = macd.macd_signal()
        df_processed['MACD_HISTOGRAM'] = macd.macd_diff()
        
        # ===== 7. ОБЪЕМ (2 признака) =====
        df_processed['VOLUME_RATIO_20'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=20).mean()
        )
        df_processed['VOLUME_RATIO_50'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=50).mean()
        )
        df_processed['VOLUME_RATIO_200'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=200).mean()
        )  # Долгосрочный объем
        
        # ===== 9. ПРОЦЕНТНЫЕ ИЗМЕНЕНИЯ =====
        df_processed['RETURN_1D'] = df_processed['close'].pct_change(5)    # Дневное изменение
        df_processed['RETURN_1W'] = df_processed['close'].pct_change(20)    # Недельное изменение (5 торговых дней)
        df_processed['RETURN_1M'] = df_processed['close'].pct_change(40)    # Месячное изменение
        
        # ===== 10. РАЗНОСТИ МЕЖДУ РАЗНЫМИ ТИПАМИ СКОЛЬЗЯЩИХ СРЕДНИХ =====
        
        # Разности между MA и EMA (разная чувствительность)
        df_processed['MA_EMA_DIFF_20'] = df_processed['MA_20'] - df_processed['EMA_20']
        df_processed['MA_EMA_DIFF_50'] = df_processed['MA_50'] - df_processed['EMA_50']
        df_processed['MA_EMA_DIFF_90'] = df_processed['MA_90'] - df_processed['EMA_90']
        df_processed['MA_EMA_DIFF_200'] = df_processed['MA_200'] - df_processed['EMA_200']  # Долгосрочная разность
        
        # Разности между MA и WMA (разные веса)
        df_processed['MA_WMA_DIFF_20'] = df_processed['MA_20'] - df_processed['WMA_20']
        df_processed['MA_WMA_DIFF_50'] = df_processed['MA_50'] - df_processed['WMA_50']
        df_processed['MA_WMA_DIFF_90'] = df_processed['MA_90'] - df_processed['WMA_90']
        df_processed['MA_WMA_DIFF_200'] = df_processed['MA_200'] - df_processed['WMA_200']  # Долгосрочная разность
        
        # Разности между EMA и WMA
        df_processed['EMA_WMA_DIFF_20'] = df_processed['EMA_20'] - df_processed['WMA_20']
        df_processed['EMA_WMA_DIFF_50'] = df_processed['EMA_50'] - df_processed['WMA_50']
        df_processed['EMA_WMA_DIFF_90'] = df_processed['EMA_90'] - df_processed['WMA_90']
        df_processed['EMA_WMA_DIFF_200'] = df_processed['EMA_200'] - df_processed['WMA_200']  # Долгосрочная разность
        
        # Отношения между разными типами скользящих средних
        df_processed['MA_EMA_RATIO_20'] = df_processed['MA_20'] / df_processed['EMA_20']
        df_processed['MA_EMA_RATIO_50'] = df_processed['MA_50'] / df_processed['EMA_50']
        df_processed['MA_EMA_RATIO_90'] = df_processed['MA_90'] / df_processed['EMA_90']
        df_processed['MA_EMA_RATIO_200'] = df_processed['MA_200'] / df_processed['EMA_200']  # Долгосрочное отношение
        
        # Разности между MA разных периодов
        df_processed['MA_DIFF_20_50'] = df_processed['MA_20'] - df_processed['MA_50']
        df_processed['MA_DIFF_20_90'] = df_processed['MA_20'] - df_processed['MA_90']
        df_processed['MA_DIFF_50_90'] = df_processed['MA_50'] - df_processed['MA_90']
        df_processed['MA_DIFF_20_200'] = df_processed['MA_20'] - df_processed['MA_200']  # Короткий vs долгий
        df_processed['MA_DIFF_50_200'] = df_processed['MA_50'] - df_processed['MA_200']  # Средний vs долгий
        
        # Разности между EMA разных периодов
        df_processed['EMA_DIFF_20_50'] = df_processed['EMA_20'] - df_processed['EMA_50']
        df_processed['EMA_DIFF_20_90'] = df_processed['EMA_20'] - df_processed['EMA_90']
        df_processed['EMA_DIFF_50_90'] = df_processed['EMA_50'] - df_processed['EMA_90']
        df_processed['EMA_DIFF_20_200'] = df_processed['EMA_20'] - df_processed['EMA_200']  # Короткий vs долгий
        df_processed['EMA_DIFF_50_200'] = df_processed['EMA_50'] - df_processed['EMA_200']  # Средний vs долгий
        
        # Разности между WMA разных периодов
        df_processed['WMA_DIFF_20_50'] = df_processed['WMA_20'] - df_processed['WMA_50']
        df_processed['WMA_DIFF_20_90'] = df_processed['WMA_20'] - df_processed['WMA_90']
        df_processed['WMA_DIFF_50_90'] = df_processed['WMA_50'] - df_processed['WMA_90']
        df_processed['WMA_DIFF_20_200'] = df_processed['WMA_20'] - df_processed['WMA_200']  # Короткий vs долгий
        df_processed['WMA_DIFF_50_200'] = df_processed['WMA_50'] - df_processed['WMA_200']  # Средний vs долгий
        
        # Отношения между MA разных периодов
        df_processed['MA_RATIO_20_50'] = df_processed['MA_20'] / df_processed['MA_50']
        df_processed['MA_RATIO_20_90'] = df_processed['MA_20'] / df_processed['MA_90']
        df_processed['MA_RATIO_50_90'] = df_processed['MA_50'] / df_processed['MA_90']
        df_processed['MA_RATIO_20_200'] = df_processed['MA_20'] / df_processed['MA_200']  # Короткий vs долгий
        df_processed['MA_RATIO_50_200'] = df_processed['MA_50'] / df_processed['MA_200']  # Средний vs долгий
        
        # Расстояния до разных типов скользящих средних
        df_processed['DISTANCE_MA_20'] = (df_processed['close'] - df_processed['MA_20']) / df_processed['MA_20']
        df_processed['DISTANCE_MA_50'] = (df_processed['close'] - df_processed['MA_50']) / df_processed['MA_50']
        df_processed['DISTANCE_MA_90'] = (df_processed['close'] - df_processed['MA_90']) / df_processed['MA_90']
        df_processed['DISTANCE_MA_200'] = (df_processed['close'] - df_processed['MA_200']) / df_processed['MA_200']  # Долгосрочное расстояние
        
        df_processed['DISTANCE_EMA_20'] = (df_processed['close'] - df_processed['EMA_20']) / df_processed['EMA_20']
        df_processed['DISTANCE_EMA_50'] = (df_processed['close'] - df_processed['EMA_50']) / df_processed['EMA_50']
        df_processed['DISTANCE_EMA_90'] = (df_processed['close'] - df_processed['EMA_90']) / df_processed['EMA_90']
        df_processed['DISTANCE_EMA_200'] = (df_processed['close'] - df_processed['EMA_200']) / df_processed['EMA_200']  # Долгосрочное расстояние
        
        df_processed['DISTANCE_WMA_20'] = (df_processed['close'] - df_processed['WMA_20']) / df_processed['WMA_20']
        df_processed['DISTANCE_WMA_50'] = (df_processed['close'] - df_processed['WMA_50']) / df_processed['WMA_50']
        df_processed['DISTANCE_WMA_90'] = (df_processed['close'] - df_processed['WMA_90']) / df_processed['WMA_90']
        df_processed['DISTANCE_WMA_200'] = (df_processed['close'] - df_processed['WMA_200']) / df_processed['WMA_200']  # Долгосрочное расстояние
        
        processed_data[ticker] = df_processed
        
        base_features = 34  # базовые признаки (1-9 группы)
        derivative_features = 49  # производные признаки (10 группа)
        total_features = base_features + derivative_features
        
        print(f" {ticker}: {total_features} признаков ({base_features} базовых + {derivative_features} производных)")
    
    return processed_data

# 3 Проверка класса для пайплайна

In [44]:
# core/feature_engineering/generator.py
import pandas as pd
import numpy as np
from typing import Optional
import ta


class FeatureGenerator:
    """Feature generator for stock price data."""
    
    def generate(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Generate all technical indicators from OHLCV data.
        
        Args:
            df: DataFrame with columns ['begin', 'open', 'high', 'low', 'close', 'volume']
                Must be sorted by 'begin'
        
        Returns:
            DataFrame with original data + generated features
        """
        if df.empty:
            return df
        
        # Создаем копию и конвертируем begin в datetime если это строка
        df_processed = df.copy()
        if df_processed['begin'].dtype == 'object':
            df_processed['begin'] = pd.to_datetime(df_processed['begin'])
        
        df_processed = df_processed.sort_values('begin')
        
        # ===== 1. MOVING AVERAGES =====
        df_processed['MA_20'] = df_processed['close'].rolling(window=20).mean()
        df_processed['MA_50'] = df_processed['close'].rolling(window=50).mean()
        df_processed['MA_90'] = df_processed['close'].rolling(window=90).mean()
        df_processed['MA_200'] = df_processed['close'].rolling(window=200).mean()
        
        df_processed['EMA_20'] = df_processed['close'].ewm(span=20).mean()
        df_processed['EMA_50'] = df_processed['close'].ewm(span=50).mean()
        df_processed['EMA_90'] = df_processed['close'].ewm(span=90).mean()
        df_processed['EMA_200'] = df_processed['close'].ewm(span=200).mean()
        
        df_processed['WMA_20'] = df_processed['close'].rolling(window=20).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_50'] = df_processed['close'].rolling(window=50).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_90'] = df_processed['close'].rolling(window=90).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        df_processed['WMA_200'] = df_processed['close'].rolling(window=200).apply(
            lambda x: np.average(x, weights=np.arange(1, len(x)+1)), raw=True
        )
        
        # ===== 2. RSI =====
        df_processed['RSI_7'] = ta.momentum.RSIIndicator(df_processed['close'], window=7).rsi()
        df_processed['RSI_14'] = ta.momentum.RSIIndicator(df_processed['close'], window=14).rsi()
        df_processed['RSI_90'] = ta.momentum.RSIIndicator(df_processed['close'], window=90).rsi()
        
        # ===== 3. MOMENTUM =====
        df_processed['MOM_10'] = ta.momentum.ROCIndicator(df_processed['close'], window=5).roc()
        df_processed['MOM_20'] = ta.momentum.ROCIndicator(df_processed['close'], window=10).roc()
        df_processed['MOM_50'] = ta.momentum.ROCIndicator(df_processed['close'], window=20).roc()
        df_processed['MOM_200'] = ta.momentum.ROCIndicator(df_processed['close'], window=200).roc()
        
        # ===== 4. VOLATILITY =====
        returns = df_processed['close'].pct_change()
        df_processed['VOLATILITY_10'] = returns.rolling(window=10).std() * np.sqrt(252)
        df_processed['VOLATILITY_20'] = returns.rolling(window=20).std() * np.sqrt(252)
        df_processed['VOLATILITY_50'] = returns.rolling(window=50).std() * np.sqrt(252)
        df_processed['VOLATILITY_200'] = returns.rolling(window=200).std() * np.sqrt(252)
        
        # ===== 5. ATR =====
        df_processed['ATR_7'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=7
        ).average_true_range()
        df_processed['ATR_14'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=14
        ).average_true_range()
        df_processed['ATR_200'] = ta.volatility.AverageTrueRange(
            df_processed['high'], df_processed['low'], df_processed['close'], window=200
        ).average_true_range()
        
        # ===== 6. MACD =====
        macd = ta.trend.MACD(df_processed['close'])
        df_processed['MACD'] = macd.macd()
        df_processed['MACD_SIGNAL'] = macd.macd_signal()
        df_processed['MACD_HISTOGRAM'] = macd.macd_diff()
        
        # ===== 7. VOLUME =====
        df_processed['VOLUME_RATIO_20'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=20).mean()
        )
        df_processed['VOLUME_RATIO_50'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=50).mean()
        )
        df_processed['VOLUME_RATIO_200'] = (
            df_processed['volume'] / df_processed['volume'].rolling(window=200).mean()
        )
        
        # ===== 8. PRICE RETURNS =====
        df_processed['RETURN_1D'] = df_processed['close'].pct_change(5)
        df_processed['RETURN_1W'] = df_processed['close'].pct_change(20)
        df_processed['RETURN_1M'] = df_processed['close'].pct_change(40)
        
        # ===== 9. DERIVATIVE FEATURES =====
        # MA-EMA differences
        df_processed['MA_EMA_DIFF_20'] = df_processed['MA_20'] - df_processed['EMA_20']
        df_processed['MA_EMA_DIFF_50'] = df_processed['MA_50'] - df_processed['EMA_50']
        df_processed['MA_EMA_DIFF_90'] = df_processed['MA_90'] - df_processed['EMA_90']
        df_processed['MA_EMA_DIFF_200'] = df_processed['MA_200'] - df_processed['EMA_200']
        
        # MA-WMA differences
        df_processed['MA_WMA_DIFF_20'] = df_processed['MA_20'] - df_processed['WMA_20']
        df_processed['MA_WMA_DIFF_50'] = df_processed['MA_50'] - df_processed['WMA_50']
        df_processed['MA_WMA_DIFF_90'] = df_processed['MA_90'] - df_processed['WMA_90']
        df_processed['MA_WMA_DIFF_200'] = df_processed['MA_200'] - df_processed['WMA_200']
        
        # EMA-WMA differences
        df_processed['EMA_WMA_DIFF_20'] = df_processed['EMA_20'] - df_processed['WMA_20']
        df_processed['EMA_WMA_DIFF_50'] = df_processed['EMA_50'] - df_processed['WMA_50']
        df_processed['EMA_WMA_DIFF_90'] = df_processed['EMA_90'] - df_processed['WMA_90']
        df_processed['EMA_WMA_DIFF_200'] = df_processed['EMA_200'] - df_processed['WMA_200']
        
        # MA-EMA ratios
        df_processed['MA_EMA_RATIO_20'] = df_processed['MA_20'] / df_processed['EMA_20']
        df_processed['MA_EMA_RATIO_50'] = df_processed['MA_50'] / df_processed['EMA_50']
        df_processed['MA_EMA_RATIO_90'] = df_processed['MA_90'] / df_processed['EMA_90']
        df_processed['MA_EMA_RATIO_200'] = df_processed['MA_200'] / df_processed['EMA_200']
        
        # MA period differences
        df_processed['MA_DIFF_20_50'] = df_processed['MA_20'] - df_processed['MA_50']
        df_processed['MA_DIFF_20_90'] = df_processed['MA_20'] - df_processed['MA_90']
        df_processed['MA_DIFF_50_90'] = df_processed['MA_50'] - df_processed['MA_90']
        df_processed['MA_DIFF_20_200'] = df_processed['MA_20'] - df_processed['MA_200']
        df_processed['MA_DIFF_50_200'] = df_processed['MA_50'] - df_processed['MA_200']
        
        # EMA period differences
        df_processed['EMA_DIFF_20_50'] = df_processed['EMA_20'] - df_processed['EMA_50']
        df_processed['EMA_DIFF_20_90'] = df_processed['EMA_20'] - df_processed['EMA_90']
        df_processed['EMA_DIFF_50_90'] = df_processed['EMA_50'] - df_processed['EMA_90']
        df_processed['EMA_DIFF_20_200'] = df_processed['EMA_20'] - df_processed['EMA_200']
        df_processed['EMA_DIFF_50_200'] = df_processed['EMA_50'] - df_processed['EMA_200']
        
        # WMA period differences
        df_processed['WMA_DIFF_20_50'] = df_processed['WMA_20'] - df_processed['WMA_50']
        df_processed['WMA_DIFF_20_90'] = df_processed['WMA_20'] - df_processed['WMA_90']
        df_processed['WMA_DIFF_50_90'] = df_processed['WMA_50'] - df_processed['WMA_90']
        df_processed['WMA_DIFF_20_200'] = df_processed['WMA_20'] - df_processed['WMA_200']
        df_processed['WMA_DIFF_50_200'] = df_processed['WMA_50'] - df_processed['WMA_200']
        
        # MA period ratios
        df_processed['MA_RATIO_20_50'] = df_processed['MA_20'] / df_processed['MA_50']
        df_processed['MA_RATIO_20_90'] = df_processed['MA_20'] / df_processed['MA_90']
        df_processed['MA_RATIO_50_90'] = df_processed['MA_50'] / df_processed['MA_90']
        df_processed['MA_RATIO_20_200'] = df_processed['MA_20'] / df_processed['MA_200']
        df_processed['MA_RATIO_50_200'] = df_processed['MA_50'] / df_processed['MA_200']
        
        # Distance to moving averages
        df_processed['DISTANCE_MA_20'] = (df_processed['close'] - df_processed['MA_20']) / df_processed['MA_20']
        df_processed['DISTANCE_MA_50'] = (df_processed['close'] - df_processed['MA_50']) / df_processed['MA_50']
        df_processed['DISTANCE_MA_90'] = (df_processed['close'] - df_processed['MA_90']) / df_processed['MA_90']
        df_processed['DISTANCE_MA_200'] = (df_processed['close'] - df_processed['MA_200']) / df_processed['MA_200']
        
        df_processed['DISTANCE_EMA_20'] = (df_processed['close'] - df_processed['EMA_20']) / df_processed['EMA_20']
        df_processed['DISTANCE_EMA_50'] = (df_processed['close'] - df_processed['EMA_50']) / df_processed['EMA_50']
        df_processed['DISTANCE_EMA_90'] = (df_processed['close'] - df_processed['EMA_90']) / df_processed['EMA_90']
        df_processed['DISTANCE_EMA_200'] = (df_processed['close'] - df_processed['EMA_200']) / df_processed['EMA_200']
        
        df_processed['DISTANCE_WMA_20'] = (df_processed['close'] - df_processed['WMA_20']) / df_processed['WMA_20']
        df_processed['DISTANCE_WMA_50'] = (df_processed['close'] - df_processed['WMA_50']) / df_processed['WMA_50']
        df_processed['DISTANCE_WMA_90'] = (df_processed['close'] - df_processed['WMA_90']) / df_processed['WMA_90']
        df_processed['DISTANCE_WMA_200'] = (df_processed['close'] - df_processed['WMA_200']) / df_processed['WMA_200']
        
        return df_processed
    
    def add_targets(self, df: pd.DataFrame, horizon_days: int = 7) -> pd.DataFrame:
        """
        Add target variables for price prediction.
        
        Args:
            df: DataFrame with 'begin' and 'close' columns
            horizon_days: Prediction horizon in days
        
        Returns:
            DataFrame with added target columns
        """
        if df.empty:
            return df
        
        # Создаем копию и конвертируем begin в datetime если это строка
        df_sorted = df.copy()
        if df_sorted['begin'].dtype == 'object':
            df_sorted['begin'] = pd.to_datetime(df_sorted['begin'])
        
        df_sorted = df_sorted.sort_values('begin')
        
        # Create price dictionary for fast lookup
        price_dict = df_sorted.set_index('begin')['close'].to_dict()
        
        # Add future date
        df_sorted['future_date'] = df_sorted['begin'] + pd.Timedelta(days=horizon_days)
        
        # Map future price
        df_sorted['future_price'] = df_sorted['future_date'].map(price_dict)
        
        # Calculate target variables
        df_sorted['target_price_change'] = (
            (df_sorted['future_price'] / df_sorted['close'] - 1) * 100
        )
        
        df_sorted['target_class'] = df_sorted['target_price_change'].apply(
            lambda x: 'L' if x < -1 else ('H' if x > 1 else 'N')
        )
        
        # Remove helper columns
        df_sorted = df_sorted.drop(['future_date', 'future_price'], axis=1)
        
        return df_sorted
    
    def clean_features(self, df: pd.DataFrame) -> pd.DataFrame:
        """
        Remove rows with NaN values after feature generation.
        
        Args:
            df: DataFrame with possible NaN values
        
        Returns:
            Cleaned DataFrame without NaN rows
        """
        return df.dropna().copy()
    
    def process(
        self,
        df: pd.DataFrame,
        include_original: bool = False,
        add_targets: bool = False,
        horizon_days: int = 7,
        clean: bool = True
    ) -> pd.DataFrame:
        """
        Main processing pipeline with configurable options.
        
        Args:
            df: Input DataFrame with OHLCV data
            include_original: Include original OHLCV columns in output
            add_targets: Add target variables for prediction
            horizon_days: Prediction horizon if add_targets=True
            clean: Remove rows with NaN values
        
        Returns:
            Processed DataFrame with selected columns
        """
        # Generate all features
        result = self.generate(df)
        
        # Add targets if requested
        if add_targets:
            result = self.add_targets(result, horizon_days)
        
        # Clean NaN values if requested
        if clean:
            result = self.clean_features(result)
        
        # Select columns to return
        if not include_original:
            # Always keep 'begin', remove original OHLCV
            original_cols = ['open', 'high', 'low', 'close', 'volume', 'value', 'ticker']
            cols_to_keep = [col for col in result.columns if col not in original_cols]
            result = result[cols_to_keep]
        
        return result

In [100]:
data_features_sber = pd.read_csv("../../data/stock_features_data/stocks_features_SBER.csv").drop(['close'], axis=1)
data_features_sber['begin'] = pd.to_datetime(data_features_sber['begin'])
data_features_sber.head(2)

,begin,MA_20,MA_50,MA_90,MA_200,EMA_20,EMA_50,EMA_90,EMA_200,WMA_20,...,DISTANCE_EMA_20,DISTANCE_EMA_50,DISTANCE_EMA_90,DISTANCE_EMA_200,DISTANCE_WMA_20,DISTANCE_WMA_50,DISTANCE_WMA_90,DISTANCE_WMA_200,target_price_change,target_class
0,2022-07-04 10:00:00,132.6825,133.4456,127.631444,124.63330,131.580771,131.515338,129.399081,126.78675,130.458762,...,-0.014826,-0.014335,0.001785,0.022425,-0.006353,-0.030611,-0.011263,0.025350,1.35771,H
1,2022-07-04 12:00:00,132.1140,133.5142,127.749222,124.64805,131.268317,131.389207,129.374651,126.80411,130.041381,...,-0.022613,-0.023512,-0.008307,0.011797,-0.013391,-0.039106,-0.021517,0.014538,0.20265,N


In [78]:
data_sber = pd.read_csv("../../data/stock_prices/stocks_SBER.csv")
data_sber.head(2)

,begin,open,close,high,low,volume,value,ticker
0,2022-05-04 10:00:00,129.10,123.95,131.50,123.6,31301400,3.977375e+09,SBER
1,2022-05-04 12:00:00,123.89,125.35,126.23,123.6,10077140,1.261006e+09,SBER


In [50]:
generator = FeatureGenerator()
test = generator.process(data_sber, add_targets=True)

In [122]:
test.head(2)

,begin,MA_20,MA_50,MA_90,MA_200,EMA_20,EMA_50,EMA_90,EMA_200,WMA_20,...,DISTANCE_EMA_20,DISTANCE_EMA_50,DISTANCE_EMA_90,DISTANCE_EMA_200,DISTANCE_WMA_20,DISTANCE_WMA_50,DISTANCE_WMA_90,DISTANCE_WMA_200,target_price_change,target_class
200,2022-07-04 10:00:00,132.6825,133.4456,127.631444,124.63330,131.580771,131.515338,129.399081,126.78675,130.458762,...,-0.014826,-0.014335,0.001785,0.022425,-0.006353,-0.030611,-0.011263,0.025350,1.35771,H
201,2022-07-04 12:00:00,132.1140,133.5142,127.749222,124.64805,131.268317,131.389207,129.374651,126.80411,130.041381,...,-0.022613,-0.023512,-0.008307,0.011797,-0.013391,-0.039106,-0.021517,0.014538,0.20265,N


In [124]:
data_features_sber.head(2)

,begin,MA_20,MA_50,MA_90,MA_200,EMA_20,EMA_50,EMA_90,EMA_200,WMA_20,...,DISTANCE_EMA_20,DISTANCE_EMA_50,DISTANCE_EMA_90,DISTANCE_EMA_200,DISTANCE_WMA_20,DISTANCE_WMA_50,DISTANCE_WMA_90,DISTANCE_WMA_200,target_price_change,target_class
0,2022-07-04 10:00:00,132.6825,133.4456,127.631444,124.63330,131.580771,131.515338,129.399081,126.78675,130.458762,...,-0.014826,-0.014335,0.001785,0.022425,-0.006353,-0.030611,-0.011263,0.025350,1.35771,H
1,2022-07-04 12:00:00,132.1140,133.5142,127.749222,124.64805,131.268317,131.389207,129.374651,126.80411,130.041381,...,-0.022613,-0.023512,-0.008307,0.011797,-0.013391,-0.039106,-0.021517,0.014538,0.20265,N


In [114]:
test.shape

(4218, 86)

In [116]:
data_features_sber.shape

(4178, 86)

In [173]:
for i in range(4178):
    MAX = 0
    m = (df1.select_dtypes(['float']).iloc[i] - df2.select_dtypes(['float']).iloc[i]).max() 
    if m > MAX:
        MAX = m

In [175]:
m

1.7763568394002505e-15